<a href="https://colab.research.google.com/github/Rammy-Saana1/2x2x2-Rubiks-Cube-Solver/blob/main/stock_forecasting_custom_colab_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# stock_forecasting_custom_colab.py
# Project: Stock Market Analysis and Forecasting Using Deep Learning
# Author: Abdul-Razak Rahama Saana
# Date: July 08, 2025
# Environment: Google Colab

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for saving plots
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import os
import sys
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)  # Suppress yfinance warning

# Install dependencies
!pip install openpyxl pillow

# Ensure output directory exists
output_dir = "/content/final_project_ml"
try:
    os.makedirs(output_dir, exist_ok=True)
except Exception as e:
    print(f"Error creating output directory: {e}")
    sys.exit(1)

# Mount Google Drive and create output directory mimicking local path
from google.colab import drive
drive.mount('/content/drive')
drive_output_dir = "/content/drive/MyDrive/Final Project ML"
os.makedirs(drive_output_dir, exist_ok=True)

# For downloading files
from google.colab import files

# 1. Data Collection
def fetch_data(ticker, start_date, end_date):
    try:
        df = yf.download(ticker, start=start_date, end=end_date, progress=False, auto_adjust=False)
        if df.empty:
            raise ValueError(f"No data for ticker {ticker}")
        # Calculate 7-day SMA
        df['SMA7'] = df['Close'].rolling(window=7).mean().fillna(method='bfill')
        # Calculate 14-day RSI
        delta = df['Close'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
        rs = gain / loss
        df['RSI14'] = 100 - (100 / (1 + rs))
        df['RSI14'] = df['RSI14'].fillna(method='bfill')
        # Include Volume
        df['Volume'] = df['Volume'].fillna(method='bfill')
        return df[['Close', 'SMA7', 'RSI14', 'Volume']].values, df.index
    except Exception as e:
        print(f"Data fetch error: {e}")
        sys.exit(1)

ticker = 'AAPL'
start_date = '2020-01-01'
end_date = '2025-07-08'
data, dates = fetch_data(ticker, start_date, end_date)

# 2. Visualize Dataset
try:
    plt.figure(figsize=(10, 5))
    plt.plot(dates, data[:, 0], label='Apple Closing Prices', color='blue')
    plt.plot(dates, data[:, 1], label='7-Day SMA', color='orange')
    plt.title('Dataset: Apple Stock Prices and 7-Day SMA (2020–2025)')
    plt.xlabel('Date')
    plt.ylabel('Price (USD)')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, 'dataset.pdf'))
    plt.savefig(os.path.join(drive_output_dir, 'dataset.pdf'))
    plt.close()
    print("Saved dataset.pdf")
    files.download(os.path.join(output_dir, 'dataset.pdf'))
except Exception as e:
    print(f"Dataset plot error: {e}")

# Salient Features
print("Salient Features:")
print(f"- Data points: {len(data)}")
print(f"- Period: {start_date} to {end_date}")
print(f"- Features: Daily closing prices, 7-day SMA, 14-day RSI, Volume, volatile due to market events")
print(f"- Source: Yahoo Finance (https://finance.yahoo.com/quote/AAPL/history/)")

# Single Sequence (30 days)
seq_length = 30
single_data = data[:seq_length, 0]
single_sma = data[:seq_length, 1]
single_dates = dates[:seq_length]
try:
    plt.figure(figsize=(10, 5))
    plt.plot(single_dates, single_data, label='Apple Closing (First 30 Days)', color='blue')
    plt.plot(single_dates, single_sma, label='7-Day SMA', color='orange')
    plt.title('Single Sequence of Stock Price and SMA')
    plt.xlabel('Date')
    plt.ylabel('Price (USD)')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, 'single_seq.pdf'))
    plt.savefig(os.path.join(drive_output_dir, 'single_seq.pdf'))
    plt.close()
    print("Saved single_seq.pdf")
    files.download(os.path.join(output_dir, 'single_seq.pdf'))
except Exception as e:
    print(f"Single sequence plot error: {e}")

# 3. Preprocessing
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)

def create_sequences(data, seq_length):
    try:
        X, y = [], []
        for i in range(len(data) - seq_length):
            X.append(data[i:i+seq_length, :])  # Include Close, SMA, RSI, Volume
            y.append(data[i+seq_length, 0])   # Predict Close only
        return np.array(X), np.array(y).reshape(-1, 1), dates[seq_length:]  # Ensure y is 2D
    except Exception as e:
        print(f"Sequence creation error: {e}")
        sys.exit(1)

X, y, seq_dates = create_sequences(data_scaled, seq_length)

# Train-Test Split
train_size = int(0.8 * len(X))
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]
test_dates = seq_dates[train_size:]

# 4. Custom LSTM-Like Model
class CustomLSTM:
    def __init__(self, input_size, hidden_size):
        self.hidden_size = hidden_size
        self.input_size = input_size
        self.Wf = np.random.randn(hidden_size, input_size + hidden_size) * 0.01
        self.Wi = np.random.randn(hidden_size, input_size + hidden_size) * 0.01
        self.Wc = np.random.randn(hidden_size, input_size + hidden_size) * 0.01
        self.Wo = np.random.randn(hidden_size, input_size + hidden_size) * 0.01
        self.Wy = np.random.randn(1, hidden_size) * 0.01
        self.bf = np.zeros((hidden_size, 1))
        self.bi = np.zeros((hidden_size, 1))
        self.bc = np.zeros((hidden_size, 1))
        self.bo = np.zeros((hidden_size, 1))
        self.by = np.zeros((1, 1))

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

    def tanh(self, x):
        return np.tanh(np.clip(x, -500, 500))

    def forward(self, X):
        predictions = []
        for x_t in X:
            h_t = np.zeros((self.hidden_size, 1))
            c_t = np.zeros((self.hidden_size, 1))
            for t in range(x_t.shape[0]):
                x_t_t = x_t[t].reshape(-1, 1)  # Input: Close, SMA, RSI, Volume
                concat = np.vstack((h_t, x_t_t))
                f_t = self.sigmoid(np.dot(self.Wf, concat) + self.bf)
                i_t = self.sigmoid(np.dot(self.Wi, concat) + self.bi)
                c_tilde = self.tanh(np.dot(self.Wc, concat) + self.bc)
                c_t = f_t * c_t + i_t * c_tilde
                o_t = self.sigmoid(np.dot(self.Wo, concat) + self.bo)
                h_t = o_t * self.tanh(c_t)
            y_pred = np.dot(self.Wy, h_t) + self.by
            predictions.append(y_pred)
        return np.array(predictions).reshape(-1, 1)

    def train(self, X, y, epochs=250, learning_rate=0.0001):
        losses = []
        for epoch in range(epochs):
            predictions = self.forward(X)
            loss = np.mean((predictions - y) ** 2)
            losses.append(loss)
            # Gradient update
            grad_Wy = np.zeros_like(self.Wy)
            grad_Wf = np.zeros_like(self.Wf)
            grad_Wi = np.zeros_like(self.Wi)
            grad_Wc = np.zeros_like(self.Wc)
            grad_Wo = np.zeros_like(self.Wo)
            grad_bf = np.zeros_like(self.bf)
            grad_bi = np.zeros_like(self.bi)
            grad_bc = np.zeros_like(self.bc)
            grad_bo = np.zeros_like(self.bo)
            for i in range(len(X)):
                x_t = X[i:i+1]
                y_true = y[i:i+1]
                pred = self.forward(x_t)
                grad = 2 * (pred - y_true) / len(X)
                h_t = np.zeros((self.hidden_size, 1))
                c_t = np.zeros((self.hidden_size, 1))
                for t in range(x_t.shape[1]):
                    x_t_t = x_t[0, t].reshape(-1, 1)
                    concat = np.vstack((h_t, x_t_t))
                    f_t = self.sigmoid(np.dot(self.Wf, concat) + self.bf)
                    i_t = self.sigmoid(np.dot(self.Wi, concat) + self.bi)
                    c_tilde = self.tanh(np.dot(self.Wc, concat) + self.bc)
                    c_t = f_t * c_t + i_t * c_tilde
                    o_t = self.sigmoid(np.dot(self.Wo, concat) + self.bo)
                    h_t = o_t * self.tanh(c_t)
                grad_Wy += np.dot(grad.T, h_t.T)
                grad_Wf += learning_rate * 0.01
                grad_Wi += learning_rate * 0.01
                grad_Wc += learning_rate * 0.01
                grad_Wo += learning_rate * 0.01
                grad_bf += learning_rate * 0.01
                grad_bi += learning_rate * 0.01
                grad_bc += learning_rate * 0.01
                grad_bo += learning_rate * 0.01
            self.Wy -= learning_rate * grad_Wy
            self.Wf -= grad_Wf
            self.Wi -= grad_Wi
            self.Wc -= grad_Wc
            self.Wo -= grad_Wo
            self.bf -= grad_bf
            self.bi -= grad_bi
            self.bc -= grad_bc
            self.bo -= grad_bo
            if epoch % 10 == 0:
                print(f"Epoch {epoch}, Loss: {loss:.4f}")
        return losses

# Train LSTM
try:
    model = CustomLSTM(input_size=4, hidden_size=256)  # Input size = 4 (Close, SMA, RSI, Volume)
    losses = model.train(X_train, y_train, epochs=250)
except Exception as e:
    print(f"LSTM training error: {e}")
    sys.exit(1)

# 5. Training Curves
try:
    plt.figure(figsize=(10, 5))
    plt.plot(losses, label='Training Loss', color='blue')
    plt.title('Training Loss Over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Mean Squared Error')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, 'training_curve.pdf'))
    plt.savefig(os.path.join(drive_output_dir, 'training_curve.pdf'))
    plt.close()
    print("Saved training_curve.pdf")
    files.download(os.path.join(output_dir, 'training_curve.pdf'))
except Exception as e:
    print(f"Training curve plot error: {e}")

# 6. Forecasting
try:
    predictions_scaled = model.forward(X_test)
    # Reshape for inverse_transform (4 features: Close, SMA, RSI, Volume)
    predictions_padded = np.hstack([predictions_scaled, np.zeros((predictions_scaled.shape[0], 3))])
    y_test_padded = np.hstack([y_test, np.zeros((y_test.shape[0], 3))])
    predictions = scaler.inverse_transform(predictions_padded)[:, 0]
    y_test_actual = scaler.inverse_transform(y_test_padded)[:, 0]
except Exception as e:
    print(f"Forecasting error: {e}")
    sys.exit(1)

# 7. Evaluation
try:
    mae = np.mean(np.abs(predictions - y_test_actual))
    rmse = np.sqrt(np.mean((predictions - y_test_actual) ** 2))
    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    metrics_df = pd.DataFrame({
        'Metric': ['MAE', 'RMSE'],
        'Value': [mae, rmse]
    })
    metrics_df.to_excel(os.path.join(output_dir, 'evaluation_metrics.xlsx'), index=False)
    metrics_df.to_excel(os.path.join(drive_output_dir, 'evaluation_metrics.xlsx'), index=False)
    print("Saved evaluation_metrics.xlsx")
    files.download(os.path.join(output_dir, 'evaluation_metrics.xlsx'))
except Exception as e:
    print(f"Metrics Excel save error: {e}")

# 8. Visualize Predictions
try:
    plt.figure(figsize=(10, 5))
    plt.plot(test_dates, y_test_actual, label='Actual Prices', color='blue')
    plt.plot(test_dates, predictions, label='Predicted Prices', color='orange')
    plt.title('Predicted vs. Actual Stock Prices')
    plt.xlabel('Date')
    plt.ylabel('Price (USD)')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, 'predictions.pdf'))
    plt.savefig(os.path.join(drive_output_dir, 'predictions.pdf'))
    plt.close()
    print("Saved predictions.pdf")
    files.download(os.path.join(output_dir, 'predictions.pdf'))
except Exception as e:
    print(f"Predictions plot error: {e}")

# 9. Single Sequence Inference
try:
    single_idx = 0
    single_sequence = X_test[single_idx:single_idx+1]
    single_pred_scaled = model.forward(single_sequence)
    single_pred_padded = np.hstack([single_pred_scaled, np.zeros((single_pred_scaled.shape[0], 3))])
    single_pred = scaler.inverse_transform(single_pred_padded)[:, 0]
    single_actual_padded = np.hstack([y_test[single_idx:single_idx+1], np.zeros((1, 3))])
    single_actual = scaler.inverse_transform(single_actual_padded)[:, 0]
    plt.figure(figsize=(10, 5))
    plt.plot([test_dates[single_idx]], single_actual, 'o', label='Actual Price', color='blue')
    plt.plot([test_dates[single_idx]], single_pred, 'x', label='Predicted Price', color='red')
    plt.title('Inference for a Single Sequence')
    plt.xlabel('Date')
    plt.ylabel('Price (USD)')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, 'single_inference.pdf'))
    plt.savefig(os.path.join(drive_output_dir, 'single_inference.pdf'))
    plt.close()
    print("Saved single_inference.pdf")
    files.download(os.path.join(output_dir, 'single_inference.pdf'))
except Exception as e:
    print(f"Single inference plot error: {e}")

# 10. 3 Correct and 3 Incorrect Predictions
try:
    errors = np.abs(predictions - y_test_actual) / y_test_actual * 100
    correct_idx = np.where(errors.flatten() <= 5)[0]
    incorrect_idx = np.where(errors.flatten() > 5)[0]
    if len(correct_idx) < 3:
        print(f"Warning: Only {len(correct_idx)} predictions with error <= 5%. Consider increasing epochs or adding features.")
        correct_idx = correct_idx[:3]
    else:
        correct_idx = correct_idx[:3]
    if len(incorrect_idx) < 3:
        print(f"Warning: Only {len(incorrect_idx)} predictions with error > 5%. Consider adjusting model.")
        incorrect_idx = incorrect_idx[-3:]
    else:
        incorrect_idx = incorrect_idx[-3:]
    results = []
    for idx in list(correct_idx) + list(incorrect_idx):
        if idx < len(test_dates):
            results.append({
                'Date': test_dates[idx].strftime('%Y-%m-%d'),
                'Actual Price': round(y_test_actual[idx], 2),
                'Predicted Price': round(predictions[idx], 2),
                'Error (%)': round(errors[idx], 2),
                'Status': 'Correct' if idx in correct_idx else 'Incorrect'
            })
    results_df = pd.DataFrame(results)
    results_df.to_excel(os.path.join(output_dir, 'correct_incorrect_predictions.xlsx'), index=False)
    results_df.to_excel(os.path.join(drive_output_dir, 'correct_incorrect_predictions.xlsx'), index=False)
    print("Correct and Incorrect Predictions:")
    print(results_df)
    print("Saved correct_incorrect_predictions.xlsx")
    files.download(os.path.join(output_dir, 'correct_incorrect_predictions.xlsx'))
    plt.figure(figsize=(10, 5))
    plt.plot(test_dates, y_test_actual, label='Actual Prices', color='blue')
    if len(correct_idx) > 0:
        plt.scatter(test_dates[correct_idx], predictions[correct_idx], c='green', label='Correct Predictions')
    if len(incorrect_idx) > 0:
        plt.scatter(test_dates[incorrect_idx], predictions[incorrect_idx], c='red', label='Incorrect Predictions')
    plt.title('Correct and Incorrect Predictions')
    plt.xlabel('Date')
    plt.ylabel('Price (USD)')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, 'correct_incorrect.pdf'))
    plt.savefig(os.path.join(drive_output_dir, 'correct_incorrect.pdf'))
    plt.close()
    print("Saved correct_incorrect.pdf")
    files.download(os.path.join(output_dir, 'correct_incorrect.pdf'))
except Exception as e:
    print(f"Correct/incorrect error: {e}")

# 11. Alternative Model (Feedforward NN)
class CustomFeedforwardNN:
    def __init__(self, input_size, hidden_size):
        self.W1 = np.random.randn(hidden_size, input_size) * 0.01
        self.b1 = np.zeros((hidden_size, 1))
        self.W2 = np.random.randn(1, hidden_size) * 0.01
        self.b2 = np.zeros((1, 1))

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

    def forward(self, X):
        predictions = []
        for x_t in X:
            x_t = x_t.flatten().reshape(-1, 1)
            h1 = self.sigmoid(np.dot(self.W1, x_t) + self.b1)
            y_pred = np.dot(self.W2, h1) + self.b2
            predictions.append(y_pred)
        return np.array(predictions).reshape(-1, 1)

    def train(self, X, y, epochs=250, learning_rate=0.0001):
        losses = []
        for epoch in range(epochs):
            predictions = self.forward(X)
            loss = np.mean((predictions - y) ** 2)
            losses.append(loss)
            grad_W2 = np.zeros_like(self.W2)
            grad_W1 = np.zeros_like(self.W1)
            grad_b1 = np.zeros_like(self.b1)
            grad_b2 = np.zeros_like(self.b2)
            for i in range(len(X)):
                x_t = X[i:i+1]
                y_true = y[i:i+1]
                pred = self.forward(x_t)
                grad = 2 * (pred - y_true) / len(X)
                h1 = np.zeros((self.W1.shape[0], 1))
                x_t_flat = x_t.flatten().reshape(-1, 1)
                h1 = self.sigmoid(np.dot(self.W1, x_t_flat) + self.b1)
                grad_W2 += np.dot(grad.T, h1.T)
                grad_h1 = np.dot(self.W2.T, grad)
                grad_b1 += grad_h1 * h1 * (1 - h1)
                grad_W1 += np.dot(grad_h1 * h1 * (1 - h1), x_t_flat.T)
                grad_b2 += grad
            self.W2 -= learning_rate * grad_W2
            self.W1 -= learning_rate * grad_W1
            self.b1 -= learning_rate * grad_b1
            self.b2 -= learning_rate * grad_b2
            if epoch % 10 == 0:
                print(f"Feedforward NN Epoch {epoch}, Loss: {loss:.4f}")
        return losses

# Train Feedforward NN
try:
    ff_model = CustomFeedforwardNN(input_size=seq_length * 4, hidden_size=256)
    ff_losses = ff_model.train(X_train, y_train, epochs=250)
except Exception as e:
    print(f"Feedforward NN training error: {e}")
    sys.exit(1)

try:
    plt.figure(figsize=(10, 5))
    plt.plot(ff_losses, label='Feedforward NN Training Loss', color='blue')
    plt.title('Feedforward NN Training Loss Over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Mean Squared Error')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, 'ff_training_curve.pdf'))
    plt.savefig(os.path.join(drive_output_dir, 'ff_training_curve.pdf'))
    plt.close()
    print("Saved ff_training_curve.pdf")
    files.download(os.path.join(output_dir, 'ff_training_curve.pdf'))
except Exception as e:
    print(f"Feedforward training curve plot error: {e}")

print("Project completed. All files saved in /content/final_project_ml and /content/drive/MyDrive/Final Project ML.")
print("Files have been downloaded. Move them to C:\\Users\\abdul\\OneDrive\\Documents\\Desktop\\Final Project ML on your local machine.")

Mounted at /content/drive
Saved dataset.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Salient Features:
- Data points: 1384
- Period: 2020-01-01 to 2025-07-08
- Features: Daily closing prices, 7-day SMA, 14-day RSI, Volume, volatile due to market events
- Source: Yahoo Finance (https://finance.yahoo.com/quote/AAPL/history/)
Saved single_seq.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Epoch 0, Loss: 0.2230
Epoch 10, Loss: 0.2234
Epoch 20, Loss: 0.2234
Epoch 30, Loss: 0.2211
Epoch 40, Loss: 0.2205
Epoch 50, Loss: 0.2226
Epoch 60, Loss: 0.2219
Epoch 70, Loss: 0.2239
Epoch 80, Loss: 0.2238
Epoch 90, Loss: 0.2238
Epoch 100, Loss: 0.2238
Epoch 110, Loss: 0.2238
Epoch 120, Loss: 0.2238
Epoch 130, Loss: 0.2238
Epoch 140, Loss: 0.2238
